# Multi-View VAE
## Download data


In [2]:
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torch
from torch import nn
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch import optim
import time

transform = transforms.ToTensor()

mnist_train = datasets.MNIST(root="data", train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root="data", train=False, download=True, transform=transform)

svhn_train = datasets.SVHN(root="data", split="train", download=True, transform=transform)
svhn_test = datasets.SVHN(root="data", split="test", download=True, transform=transform)


Using downloaded and verified file: data/train_32x32.mat


Using downloaded and verified file: data/test_32x32.mat


In [3]:
transform = transforms.ToTensor()

def show_samples(dataset, title, n=8):
    fig, axes = plt.subplots(1, n, figsize=(n * 1.3, 1.8))
    for ax, index in zip(axes, range(n)):
        image, label = dataset[index]
        ax.imshow(image.permute(1, 2, 0).squeeze(), cmap="gray")
        ax.set_title(int(label))
        ax.axis("off")
    fig.suptitle(f"{title}  ({len(dataset):,} images)")
    fig.tight_layout()
    plt.show()

mnist_train = datasets.MNIST(root="data", train=True, download=False, transform=transform)
mnist_test = datasets.MNIST(root="data", train=False, download=False, transform=transform)
print(f"MNIST train: {len(mnist_train):,}   test: {len(mnist_test):,}")
show_samples(mnist_train, "MNIST train")

print(f"SVHN train: {len(svhn_train):,}   test: {len(svhn_test):,}")
show_samples(svhn_train, "SVHN train")


MNIST train: 60,000   test: 10,000


SVHN train: 73,257   test: 26,032


In [4]:
# The encoder below expects a batch of 64x64 RGB images scaled to [-1, 1],
# the same format as the CelebA lab. A DataLoader is what produces that batch:
# each step yields (images, labels) with images shaped (batch_size, 3, 64, 64).
# MNIST is grayscale, so its single channel is repeated three times.

def model_transform():
    return transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Lambda(lambda image: image.repeat(3, 1, 1) if image.shape[0] == 1 else image),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

def get_loader(dataset, batch_size):
    sampler = torch.utils.data.SubsetRandomSampler(range(len(dataset)))
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, sampler=sampler)

batch_size = 32
transform_64 = model_transform()

mnist_train_model = datasets.MNIST(root="data", train=True, download=False, transform=transform_64)
svhn_train_model = datasets.SVHN(root="data", split="train", download=False, transform=transform_64)

mnist_loader = get_loader(mnist_train_model, batch_size)
svhn_loader = get_loader(svhn_train_model, batch_size)


In [5]:
def imshow(img):
    img = img / 2 + 0.5     # undo Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

mnist_batch, mnist_labels = next(iter(mnist_loader))
svhn_batch, svhn_labels = next(iter(svhn_loader))

print("MNIST batch", tuple(mnist_batch.shape), "labels", mnist_labels[:8].tolist())
print("SVHN batch", tuple(svhn_batch.shape), "labels", svhn_labels[:8].tolist())

imshow(torchvision.utils.make_grid(mnist_batch))
imshow(torchvision.utils.make_grid(svhn_batch))


MNIST batch (32, 3, 64, 64) labels [7, 0, 2, 0, 7, 9, 7, 1]
SVHN batch (32, 3, 64, 64) labels [7, 2, 1, 3, 3, 3, 8, 1]


In [6]:
class encoder(nn.Module):
    
    def __init__(self,dimz,channels=3,var_x=0.1):
        
        
        super().__init__()
        
        self.dimz = dimz    #dimz is k, the dimension of the latent space

        # self.conv1 is a convolutional layer, with 32 output channels, kernel size 4, stride 2,
        # and padding 1
        
        self.conv1 = nn.Conv2d(in_channels=channels, out_channels=32, 
                               kernel_size=4, stride=2, padding=1)
        
        self.relu = nn.ReLU()
        
        # self.conv2 is a convolutional layer, with 32 output channels, kernel size 4, stride 2,
        # and padding 1

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, 
                               kernel_size=4, stride=2, padding=1) #YOUR CODE HERE

        # self.conv3 is a convolutional layer, with 64 output channels, kernel size 4, stride 2,
        # and padding 1  
        
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, 
                               kernel_size=4, stride=2, padding=1)      

        # self.conv4 is a convolutional layer, with 64 output channels, kernel size 4, stride 2,
        # and padding 1        
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, 
                               kernel_size=4, stride=2, padding=1)  #YOUR CODE HERE

        # self.conv5 is a convolutional layer, with 256 output channels, kernel size 4, stride 1,
        # and padding 0                                
        
        self.conv5 = nn.Conv2d(in_channels=64, out_channels=256, 
                               kernel_size=4, stride=1, padding=0) 
        

        # self.linear is a linear layer with dimz*2 outputs
        self.linear = nn.Linear(256,dimz*2) #YOUR CODE HERE
        
        self.softplus = nn.Softplus()
        
    def forward(self,x):

        # The forward method to project and image into a 2dimz dimensional vector
        
        z = self.relu(self.conv1(x))
        z = self.relu(self.conv2(z)) #YOUR CODE HERE
        z = self.relu(self.conv3(z))
        z = self.relu(self.conv4(z)) #YOUR CODE HERE
        z = self.relu(self.conv5(z))
        # Transform z into a 256-dim vector
        z = z.view(-1,256) #YOUR CODE HERE
        z = self.linear(z)
        
        return z
    
    def encode_and_sample(self,x,flag_sample=True):

        # This methods compute both the posterior mean and variance
        # Also we obtain a sample from the posterior using the
        # reparameterization trick.

        # We obtain the encoder projection using the forward method
        
        z = self.forward(x) #YOUR CODE HERE

        # The mean is the first dimz components of the forward output
        
        mu = z[:, :self.dimz] #YOUR CODE HERE

        # We compute the variance from the last dimz components using a 
        # soft plus
        var = self.softplus(0.5 * z[:, self.dimz:])
        
        sample = None
        
        if(flag_sample==True):
            
            eps = torch.randn_like(var) 
            
            sample = mu + eps*(var**0.5)
        
        return mu,var,sample
    


In [7]:
my_enc = encoder(dimz=2)

mu_z, var_z, _ = my_enc.encode_and_sample(mnist_batch)
print("MNIST mu", mu_z[0])
print("MNIST var", var_z[0])

mu_z, var_z, _ = my_enc.encode_and_sample(svhn_batch)
print("SVHN mu", mu_z[0])
print("SVHN var", var_z[0])


MNIST mu tensor([-0.0279,  0.0439], grad_fn=<SelectBackward0>)
MNIST var tensor([0.6896, 0.6909], grad_fn=<SelectBackward0>)
SVHN mu tensor([-0.0291,  0.0354], grad_fn=<SelectBackward0>)
SVHN var tensor([0.6882, 0.6913], grad_fn=<SelectBackward0>)


## Decoder


In [8]:
class decoder(nn.Module):
    
    def __init__(self,dimz,channels=3,var_x=0.1):
        
        super().__init__()

        # We expand z into a 256 dimensional vector 
        
        self.linear = nn.Linear(dimz,256)
        
        self.relu = nn.ReLU()

        self.tanh = nn.Tanh()

        # self.tconv1 is a convolutional layer, with 64 output channels, kernel size 4, stride 1,
        # and padding 0 
        
        self.tconv1 = nn.ConvTranspose2d(256, 64, kernel_size=4, stride=1,padding=0)

        # self.tconv2 is a convolutional layer, with 64 output channels, kernel size 4, stride 2,
        # and padding 1         
        
        self.tconv2 = nn.ConvTranspose2d(64, 64, kernel_size=4, stride=2,padding=1) #YOUR CODE HERE

        # self.tconv3 is a convolutional layer, with 32 output channels, kernel size 4, stride 2,
        # and padding 1  
        
        self.tconv3 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2,padding=1)

        # self.tconv3 is a convolutional layer, with 32 output channels, kernel size 4, stride 2,
        # and padding 1
        
        self.tconv4 = nn.ConvTranspose2d(32, 32, kernel_size=4, stride=2,padding=1) #YOUR CODE HERE

        # self.tconv3 is a convolutional layer, with channels output channels, kernel size 4, stride 2,
        # and padding 1        
        
        self.tconv5 = nn.ConvTranspose2d(32, channels, kernel_size=4, stride=2,padding=1)
        
    def forward(self,z):

        x = self.relu(self.linear(z).view(-1,256,1,1))
        x = self.relu(self.tconv1(x)) #YOUR CODE HERE
        x = self.relu(self.tconv2(x))
        x = self.relu(self.tconv3(x)) #YOUR CODE HERE
        x = self.relu(self.tconv4(x))
        x = self.tanh(self.tconv5(x)) #YOUR CODE HERE   
        return x
    
    def decode(self,z):

        # This function simply calls the forward method
        
        return self.forward(z)


In [13]:
my_dec = encoder(dimz=2)

mu_z, var_z, _ = my_dec.encode_and_sample(mnist_batch)
print("MNIST mu", mu_z[0])
print("MNIST var", var_z[0])

my_dec = decoder(dimz=2)
x_mean = my_dec.decode(mu_z).detach()
imshow(mnist_batch[0,:,:,:])
imshow(x_mean[0,:,:,:])


my_dec = encoder(dimz=2)
mu_z, var_z, _ = my_dec.encode_and_sample(svhn_batch)
print("SVHN mu", mu_z[0])
print("SVHN var", var_z[0])

my_dec = decoder(dimz=2)
x_mean = my_dec.decode(mu_z).detach()
imshow(svhn_batch[0,:,:,:])
imshow(x_mean[0,:,:,:])


MNIST mu tensor([0.0081, 0.0403], grad_fn=<SelectBackward0>)
MNIST var tensor([0.6918, 0.6868], grad_fn=<SelectBackward0>)


SVHN mu tensor([ 0.0442, -0.0299], grad_fn=<SelectBackward0>)
SVHN var tensor([0.6921, 0.6915], grad_fn=<SelectBackward0>)


In [17]:
def eval_Gaussian_LL(x,mu_x,var_x):

    # x is a mini-batch of images. It has dimension [Batch,3,dimx,dimx]

    # mu_x is a mini-batch of reconstructed images. It has dimension [Batch,3,dimx,dimx]

    # var_x is a torch constant

    D = x.shape[1] * x.shape[2] * x.shape[3]   # Dimension of the image

    x = x.reshape(-1, D)

    mu_x = mu_x.reshape(-1, D)

    var_x = torch.ones_like(mu_x) * var_x

    # Constant term in the gaussian distribution
    cnt = D * np.log(2 * np.pi) + torch.sum(torch.log(var_x), dim=-1)

    # log-likelihood per datapoint

    logp_data = -0.5 * (cnt + torch.sum((x - mu_x) * var_x ** -1 * (x - mu_x), dim=-1))
    
    # Accumulated Gaussian log-likelihood for all datapoints in the batch
    logp = torch.sum(logp_data)

    return logp,logp_data


## VAE


In [18]:
class VAE(nn.Module):
    
    def __init__(self,dimz,channels=3,var_x=0.1):
        
        super().__init__()
        
        self.var_x = var_x
        
        self.dimz = dimz

        # We create an encoder network
        
        self.encoder = encoder(self.dimz,channels,var_x) #YOUR CODE HERE

        # We create a decoder network
        
        self.decoder = decoder(self.dimz,channels,var_x) #YOUR CODE HERE
        
    def forward(self,x):

        # In the forward method, we return the mean and variance 
        # given by the encoder network and also the reconstruction mean
        # given by the decoder network using a sample from the 
        # encoder's posterior distribution.
        
        mu_z,var_z,sample_z = self.encoder.encode_and_sample(x) #YOUR CODE HERE
        
        # Decoder provides the mean of the reconstruction
        
        mu_x = self.decoder.decode(sample_z) 
        
        return mu_x,mu_z,var_z
    
    # Reconstruction + KL divergence losses summed over all elements and batch

    def loss_function(self, x, mu_x, mu_z, var_z):

        # We evaluate the loglikelihood in the batch using the function provided above

        logp,_ = eval_Gaussian_LL(x,mu_x,self.var_x)  #YOUR CODE HERE

        # KL divergence between q(z) and N()
        # see Appendix B from VAE paper:
        # Kingma and Welling. Auto-Encoding Variational Bayes. ICLR, 2014
        # https://arxiv.org/abs/1312.6114

        KLz = -0.5 * torch.sum(1 + torch.log(var_z) - mu_z.pow(2) - var_z)

        # To maximize ELBO we minimize loss (-ELBO)
        return -logp + KLz, -logp, KLz  


In [19]:
my_vae = VAE(dimz=2)

print(my_vae.loss_function(svhn_batch,x_mean,mu_z,var_z))

my_vae = VAE(dimz=2)

print(my_vae.loss_function(mnist_batch,x_mean,mu_z,var_z))


(tensor(198716.8906, grad_fn=<AddBackward0>), tensor(198714.9219), tensor(1.9687, grad_fn=<MulBackward0>))
(tensor(1557475.7500, grad_fn=<AddBackward0>), tensor(1557473.7500), tensor(1.9687, grad_fn=<MulBackward0>))
